# Slabs of SrTiO3(011) and (001) with specified terminations

## 0. Introduction

This notebook creates the SrTiO3 slabs of the manuscript:

> **R. I. Eglitis and David Vanderbilt**
> *First-principles calculations of atomic and electronic structure of SrTiO3 (001) and (011) surfaces*
> Phys. Rev. B 77, 195408 (2008)
> [DOI: 10.1103/PhysRevB.77.195408](https://doi.org/10.1103/PhysRevB.77.195408)

The (011) planes alternate between SrTiO and O2 (FIG. 2), both charged, so a slab cut on whole planes is either polar or charged (FIG. 3(a)-(c)). The paper's three (011) slabs are seven planes thick and remove atoms from both outer planes of a symmetric slab (FIG. 3(d)-(f)):

- TiO-terminated: Sr removed from the SrTiO end planes, 16 atoms;
- Sr-terminated: Ti and O removed from the SrTiO end planes, 14 atoms;
- O-terminated: one of the two O removed from the O2 end planes, 15 atoms, the only stoichiometric one.

The (001) SrO- and TiO2-terminated slabs (FIG. 1) are seven whole planes. All slabs are saved under fixed names for the [simulation notebook](slab_strontium_titanate_SIMULATION.ipynb), together with the bulk cell.

 ![FIG. 2.](https://github.com/Exabyte-io/documentation/raw/12617167278ae3523adc028583b21ea4e8ebd197/images/tutorials/materials/2d_materials/slab_strontium_titanate/0-figure-from-manuscript.webp)

## 1. Prepare the Environment
### 1.1. Set up slab parameters

In [ ]:
NUMBER_OF_PLANES = 7  # paper: seven-plane slabs; nine change the energies by < 0.01 eV
THICKNESS = 4  # in unit cells of two planes each: eight planes hold both seven-plane cuts
VACUUM = 10.0  # in angstroms; the vacuum left on each cut slab is printed in section 4
XY_SUPERCELL_MATRIX = [[1, 0], [0, 1]]
USE_ORTHOGONAL_C = True
USE_CONVENTIONAL_CELL = True

# termination: (Miller indices, formula of the two end planes, atoms removed from each end plane)
SLAB_TERMINATIONS = {
    "TiO": ((0, 1, 1), "SrTiO", ["Sr"]),  # FIG. 3(d)
    "Sr": ((0, 1, 1), "SrTiO", ["Ti", "O"]),  # FIG. 3(e)
    "O": ((0, 1, 1), "O2", ["O"]),  # FIG. 3(f)
    "SrTiO": ((0, 1, 1), "SrTiO", []),  # FIG. 3(c), charged, not computed in the paper
    "O2": ((0, 1, 1), "O2", []),  # FIG. 3(b), charged, not computed in the paper
    "SrO": ((0, 0, 1), "SrO", []),  # FIG. 1(a)
    "TiO2": ((0, 0, 1), "TiO2", []),  # FIG. 1(b)
}

### 1.2. Install Packages
The step executes only in Pyodide environment. For other environments, the packages should be installed via `pip install` (see [README](../../README.ipynb)).

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples")

### 1.3. Get input materials

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials

material = Material.create(Materials.get_by_name_first_match("SrTiO3"))

### 1.4. Preview the material

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

visualize(material, repetitions=[3, 3, 3], rotation="0x")
visualize(material, repetitions=[3, 3, 3], rotation="-90x")

## 2. Get possible terminations for the slab


In [ ]:
from mat3ra.made.tools.helpers import get_slab_terminations

MILLER_INDICES = sorted({miller_indices for miller_indices, _, _ in SLAB_TERMINATIONS.values()}, reverse=True)
for miller_indices in MILLER_INDICES:
    slab_terminations = get_slab_terminations(material, miller_indices=miller_indices)
    print(f"Terminations {miller_indices}: {[str(termination) for termination in slab_terminations]}")

## 3. Create the whole-plane slabs

The builder cuts whole planes only, two per unit cell, so each slab has a different plane on each face.

In [ ]:
from mat3ra.made.tools.analyze.other import get_atom_indices_by_layer
from mat3ra.made.tools.helpers import create_slab

ELEMENTS = list(dict.fromkeys(material.basis.elements.values))


def get_formula(slab, ids):
    elements = [slab.basis.elements.values[i] for i in ids]
    return "".join(f"{e}{elements.count(e) if elements.count(e) > 1 else ''}" for e in ELEMENTS if e in elements)


whole_plane_slabs = {}
for miller_indices in MILLER_INDICES:
    slab = create_slab(
        crystal=material,
        miller_indices=miller_indices,
        number_of_layers=THICKNESS,
        vacuum=VACUUM,
        use_orthogonal_c=USE_ORTHOGONAL_C,
        use_conventional_cell=USE_CONVENTIONAL_CELL,
        xy_supercell_matrix=XY_SUPERCELL_MATRIX,
    )
    whole_plane_slabs[miller_indices] = slab
    planes = get_atom_indices_by_layer(slab)
    print(f"{miller_indices}, bottom to top: {' | '.join(get_formula(slab, plane) for plane in planes)}")

## 4. Cut the seven-plane slabs and remove atoms from their end planes

Each slab keeps `NUMBER_OF_PLANES` consecutive planes with the given formula at both ends, then removes the listed atoms from both end planes, at the same in-plane site on each face. The nominal charge (Sr +2, Ti +4, O -2) shows why the paper trims the whole-plane (011) slabs: they carry ±4 e, the trimmed ones none. The faces are equivalent when a symmetry operation of the slab maps the top onto the bottom.

In [ ]:
from mat3ra.made.tools.convert import to_pymatgen
from mat3ra.made.tools.modify import filter_by_ids
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

NOMINAL_CHARGES = {"Sr": 2, "Ti": 4, "O": -2}


def have_equivalent_faces(slab):
    operations = SpacegroupAnalyzer(to_pymatgen(slab)).get_symmetry_operations()
    return any(operation.rotation_matrix[2][2] == -1 for operation in operations)


slabs = []
for termination, (miller_indices, end_plane_formula, removed_elements) in SLAB_TERMINATIONS.items():
    whole_plane_slab = whole_plane_slabs[miller_indices]
    planes = get_atom_indices_by_layer(whole_plane_slab)
    first = next(i for i, plane in enumerate(planes) if get_formula(whole_plane_slab, plane) == end_plane_formula)
    kept_planes = planes[first:first + NUMBER_OF_PLANES]
    coordinates = whole_plane_slab.basis.coordinates.values
    elements = whole_plane_slab.basis.elements.values
    removed_ids = [
        min((i for i in plane if elements[i] == element), key=lambda i: coordinates[i][:2])
        for plane in (kept_planes[0], kept_planes[-1])
        for element in removed_elements
    ]
    kept_ids = [i for plane in kept_planes for i in plane if i not in removed_ids]
    slab = filter_by_ids(whole_plane_slab, kept_ids, reset_ids=True)
    slab.name = f"SrTiO3({''.join(map(str, miller_indices))}) {termination}-terminated {NUMBER_OF_PLANES} planes"
    slabs.append(slab)

    heights = [z for _, _, z in slab.basis.coordinates.values]
    vacuum = slab.lattice.c * (1 - max(heights) + min(heights))
    charge = sum(NOMINAL_CHARGES[element] for element in slab.basis.elements.values)
    print(f"{slab.name}: {get_formula(slab, range(slab.basis.number_of_atoms))}, {slab.basis.number_of_atoms} atoms, "
          f"nominal charge {charge:+d} e, equivalent faces: {have_equivalent_faces(slab)}, vacuum {vacuum:.1f} Å")
    print(f"    bottom to top: {' | '.join(get_formula(slab, plane) for plane in get_atom_indices_by_layer(slab))}")

## 5. Visualize the slabs

In [ ]:
visualize([{"material": slab, "title": slab.name} for slab in slabs], repetitions=[3, 3, 1], rotation="-90x")

## 6. Download materials

In [ ]:
from mat3ra.notebooks_utils.io import download_content_to_file
from mat3ra.notebooks_utils.material import set_materials

set_materials([material, *slabs])
for slab in [material, *slabs]:
    download_content_to_file(slab.to_json(), f"{slab.name}.json")